In [ ]:
pip install -U langchain-huggingface sentence-transformers

In [4]:
import os

from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

Load environment variables

In [5]:
GROQ_API_KEY = "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"

In [6]:
llm = ChatGroq(
    #model="meta-llama/llama-prompt-guard-2-86m",
    #model="openai/gpt-oss-120b",
    model="openai/gpt-oss-20b",
    temperature = 0,
    api_key = GROQ_API_KEY
)

In [8]:
response = llm.invoke("tell me about past perfect tense")

In [9]:
response.content

'## Past Perfect Tense (also called the **pluperfect**)\n\n| **Form** | **Structure** | **Example** |\n|----------|---------------|-------------|\n| **Affirmative** | *had* + past participle | She **had finished** the report before the meeting. |\n| **Negative** | *had not* (or *hadn’t*) + past participle | He **hadn’t seen** the movie yet. |\n| **Question** | *Had* + subject + past participle? | **Had** you **already eaten** when I called? |\n\n> **Key point:** The past perfect describes an action that was **completed before another past event**.\n\n---\n\n### 1. When to Use It\n\n| Situation | Why Past Perfect? | Example |\n|-----------|-------------------|---------|\n| **Narrative** (storytelling) | Shows the order of events. | By the time the police arrived, the thief **had escaped**. |\n| **Reported speech** | Keeps the original past‑time relationship. | She said she **had finished** the work. |\n| **Conditional “if” clauses** (third‑person conditionals) | Expresses a hypothetical

In [10]:
from groq import Groq

In [11]:
documents = [
    Document(
        page_content=(
            "LangGraph is a framework for building stateful "
            "and multi-agent AI applications."
        ),
        metadata={"source": "langgraph_notes"},
    ),
       Document(
        page_content=(
            "Inception BD is a Edtech Platform "
            "and provides courses on AI."
        ),
        metadata={"source": "inception_notes"},
    ),
    Document(
        page_content=(
            "RAG stands for Retrieval-Augmented Generation. "
            "It retrieves relevant information before generating an answer."
        ),
        metadata={"source": "rag_notes"},
    ),
    Document(
        page_content=(
            "Groq provides fast inference for supported large "
            "language models through the Groq API."
        ),
        metadata={"source": "groq_notes"},
    ),
    Document(
        page_content=(
            "FAISS is a vector similarity-search library. "
            "It can retrieve documents whose embeddings are close "
            "to the query embedding."
        ),
        metadata={"source": "faiss_notes"},
    ),
]


In [12]:
type(documents)

list

Load the embedding model

In [13]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={
        "normalize_embeddings": True,
    },
)


/opt/anaconda3/envs/Agents/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1023 17:25:03.349000 2660 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7657.50it/s]


Create the FAISS vector store and store embeddings

In [16]:
pip install -U faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 10.1 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [17]:
vector_store = FAISS.from_documents(
    documents=documents,
    embedding=embeddings,
)


In [18]:
vector_store.save_local("faiss_index")

In [19]:
vector_store = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)


Create a retriever

In [22]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2},
)

In [24]:
# create the prompt

prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful assistant.
    
    Answer the question using only the provided context.

    if the answer is not present  in the context , say :
        "I don't know base on the provided context."

    Context:
    {context}

    Question:
    {question}

    Answer: 
    """
)

Format retrieved documents

In [26]:
def format_documents(retrieved_documents: list[Document]) -> str:
    return "\n\n".join(
        document.page_content
        for document in retrieved_documents
    )


   

Build the RAG Chainb


In [28]:
rag_chain= (
    {
        "context" : retriever | format_documents,
        "question": RunnablePassthrough(),

    }
    |prompt
    |llm
    |StrOutputParser()
)

Askquestion

In [30]:
def ask_question(question: str) -> str:
    if not question.strip():
        raise ValueError("question cannot be empty")
    return rag_chain.invoke(question)

In [41]:
if __name__ == "__main__":
    while True:
        user_question = input(
            "\nAsk a question or type 'exit': "
        ).strip()

        if user_question.lower() == "exit":
            print("Application closed.")
            break

        try:
            answer = ask_question(user_question)

            print("\nAnswer:")
            print(answer)

        except Exception as error:
            print(f"\nError: {error}")


Answer:
RAG stands for Retrieval‑Augmented Generation. It retrieves relevant information before generating an answer.

Answer:
Inception BD is an Edtech platform that provides courses on AI.
Application closed.


In [40]:
if __name__=="__main__":
    while True:
        user_question = input(
            "\nAsk a question or type 'exit': "
        ).strip()

        if user_question.lower() == "exit":
            print("Application closed")
            break

        try:
            anser = ask_question(user_question)

            print("\nAnser:")
            print(answer)

        except Exception as error:
            print(f"\nError: {error}")


Anser:
I don't know base on the provided context.

Anser:
I don't know base on the provided context.
Application closed
